# Stage 4 - Predict 2026 Group-Stage Fixture Probabilities

Load the trained best-model bundle and the 2026 fixture features, apply the
**exact** Stage 3 preprocessing, and produce 3-way outcome probabilities
(`P_home_win`, `P_draw`, `P_away_win`) for every fixture.

**No training, no actual 2026 scores, no tournament simulation** - only
fixture-level probabilities.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config, predict_2026

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

## 1-2. Load the trained model bundle and fixture features

In [2]:
bundle = predict_2026.load_bundle()
fixtures = pd.read_parquet(config.PROCESSED_FILES["fixtures_2026_features"])
fixtures["date"] = pd.to_datetime(fixtures["date"])
print("model       :", bundle["model_name"])
print("class order :", bundle["class_order"])
print("feature cols:", bundle["feature_names"])
print("fixtures    :", fixtures.shape)

model       : RandomForest
class order : ['home_win', 'draw', 'away_win']
feature cols: ['home_elo', 'away_elo', 'elo_diff', 'home_winrate_last5', 'away_winrate_last5', 'home_goals_for_last5', 'away_goals_for_last5', 'home_goals_against_last5', 'away_goals_against_last5', 'home_goal_diff_last5', 'away_goal_diff_last5', 'neutral', 'tournament_weight', 'home_elo_missing', 'away_elo_missing', 'elo_diff_missing']
fixtures    : (72, 17)


## 3. Apply the exact same Stage 3 preprocessing

Same feature list, same Elo missingness indicators, same train medians. We
confirm the resulting matrix has the expected columns and no NaNs.

In [3]:
X = predict_2026.apply_preprocessing(fixtures, bundle["impute_state"])
print("matrix shape:", X.shape)
print("columns match training:", list(X.columns) == bundle["feature_names"])
print("remaining NaNs:", int(X.isna().sum().sum()))
X.head()

matrix shape: (72, 16)
columns match training: True
remaining NaNs: 0


,home_elo,away_elo,elo_diff,home_winrate_last5,away_winrate_last5,home_goals_for_last5,away_goals_for_last5,home_goals_against_last5,away_goals_against_last5,home_goal_diff_last5,away_goal_diff_last5,neutral,tournament_weight,home_elo_missing,away_elo_missing,elo_diff_missing
0,1835.0,1524.0,30.0,0.6,0.2,1.8,0.8,0.4,1.0,1.4,-0.2,0,1.0,0,1,1
1,1543.0,1524.0,30.0,0.6,0.6,1.4,3.4,1.0,1.2,0.4,2.2,1,1.0,1,1,1
2,1802.0,1524.0,30.0,0.4,0.0,1.4,0.8,0.6,0.8,0.8,0.0,0,1.0,0,1,1
3,1543.0,1833.0,30.0,0.4,0.6,2.2,1.8,2.4,1.0,-0.2,0.8,0,1.0,1,0,1
4,1427.0,1897.0,-470.0,0.0,0.2,0.2,1.8,1.2,1.4,-1.0,0.4,1,1.0,0,0,0


## 4. Predict fixture probabilities

In [4]:
preds = predict_2026.predict(fixtures, bundle)
# Probabilities must sum to 1 for every fixture.
row_sums = preds[predict_2026.PROBA_COLS].sum(axis=1)
assert (row_sums.round(6) == 1.0).all()
print("OK - each fixture's probabilities sum to 1")
preds.head()

OK - each fixture's probabilities sum to 1


,date,home_team,away_team,tournament,P_home_win,P_draw,P_away_win,predicted_outcome,confidence
0,2026-06-11,Mexico,South Africa,FIFA World Cup,0.706698,0.199501,0.093801,home_win,0.706698
1,2026-06-11,South Korea,Czech Republic,FIFA World Cup,0.340167,0.311774,0.348058,away_win,0.348058
2,2026-06-12,Canada,Bosnia and Herzegovina,FIFA World Cup,0.634462,0.238828,0.126710,home_win,0.634462
3,2026-06-12,United States,Paraguay,FIFA World Cup,0.165919,0.354479,0.479601,away_win,0.479601
4,2026-06-13,Qatar,Switzerland,FIFA World Cup,0.050132,0.133375,0.816493,away_win,0.816493


## 5-6. Save predictions and print the summary report

`predict_2026.run()` writes `outputs/fixtures_2026_predictions.csv` and prints
the first 10 predictions, highest-confidence matches, most balanced matches,
and the average predicted draw probability.

In [5]:
preds = predict_2026.run()

Loaded model: RandomForest
Fixtures: 72 rows

2026 GROUP-STAGE FIXTURE PREDICTIONS
fixtures predicted: 72

--- First 10 predictions ---
      date     home_team              away_team  P_home_win  P_draw  P_away_win predicted_outcome  confidence
2026-06-11        Mexico           South Africa       0.707   0.200       0.094          home_win       0.707
2026-06-11   South Korea         Czech Republic       0.340   0.312       0.348          away_win       0.348
2026-06-12        Canada Bosnia and Herzegovina       0.634   0.239       0.127          home_win       0.634
2026-06-12 United States               Paraguay       0.166   0.354       0.480          away_win       0.480
2026-06-13         Qatar            Switzerland       0.050   0.133       0.816          away_win       0.816
2026-06-13        Brazil                Morocco       0.569   0.223       0.208          home_win       0.569
2026-06-13         Haiti               Scotland       0.126   0.250       0.624          away_

## Predicted outcome distribution

A quick look at how the most-likely outcomes break down across the 72 fixtures.

In [6]:
print(preds["predicted_outcome"].value_counts())
print("\nmean probabilities across fixtures:")
print(preds[predict_2026.PROBA_COLS].mean().round(4))

predicted_outcome
home_win    36
away_win    31
draw         5
Name: count, dtype: int64

mean probabilities across fixtures:
P_home_win    0.3773
P_draw        0.2840
P_away_win    0.3387
dtype: float64
